# CAST Phase 3 — XLM-R MAD-X Transfer (VALIDATION)



In [ ]:
# Environment setup
!pip install -q -U adapters accelerate datasets

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import adapters
print(f"adapters {adapters.__version__} ready")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 145.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive
adapters 1.3.0 ready


In [ ]:
# Configuration
import gc
import json
import os
import random
import shutil
import warnings

import adapters.composition as ac
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from adapters import SeqBnConfig, SeqBnInvConfig, XLMRobertaAdapterModel
from datasets import load_dataset
from sklearn.metrics import f1_score
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SAVE_DIR = "/content/drive/MyDrive/EmotionDetection/CAST_checkpoints"
P3_DIR = f"{SAVE_DIR}/Phase3"
CACHE_DIR = f"{P3_DIR}/data_cache"
CKPT_DIR = f"{P3_DIR}/epoch_ckpt_xlmr_weighted"
ADAPTER_DIR = f"{P3_DIR}/adapters_xlmr_weighted"
PRED_DIR = f"{P3_DIR}/predictions"
MODEL_CACHE_DIR = f"{SAVE_DIR}/model_cache"
LOCAL_MODEL_CACHE = "/content/hf_cache_local"

for directory in [
    P3_DIR,
    CACHE_DIR,
    CKPT_DIR,
    ADAPTER_DIR,
    PRED_DIR,
    MODEL_CACHE_DIR,
    LOCAL_MODEL_CACHE,
]:
    os.makedirs(directory, exist_ok=True)

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{MODEL_CACHE_DIR}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{MODEL_CACHE_DIR}/datasets"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMOTION_ORDER = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LANG_ORDER = ["eng", "hin", "rus", "hau", "kin", "sun", "yor", "vmw", "pcm"]
DATASET_HF_NAME = "brighter-dataset/BRIGHTER-emotion-categories"

LANGUAGES = {
    "eng": {"name": "English", "tier": 1, "family": "Indo-European", "genus": "Germanic"},
    "hin": {"name": "Hindi", "tier": 1, "family": "Indo-European", "genus": "Indo-Aryan"},
    "rus": {"name": "Russian", "tier": 1, "family": "Indo-European", "genus": "Slavic"},
    "hau": {"name": "Hausa", "tier": 2, "family": "Afroasiatic", "genus": "Chadic"},
    "kin": {"name": "Kinyarwanda", "tier": 2, "family": "Niger-Congo", "genus": "Bantu"},
    "sun": {"name": "Sundanese", "tier": 2, "family": "Austronesian", "genus": "Sundic"},
    "yor": {"name": "Yoruba", "tier": 3, "family": "Niger-Congo", "genus": "Volta-Niger"},
    "vmw": {"name": "Emakhuwa", "tier": 3, "family": "Niger-Congo", "genus": "Bantu"},
    "pcm": {"name": "Nigerian Pidgin", "tier": 3, "family": "Creole", "genus": "English-Lexifier"},
}
if list(LANGUAGES) != LANG_ORDER:
    raise ValueError("LANGUAGES must follow LANG_ORDER.")

ALL_TARGETS = LANG_ORDER.copy()
RUN_TARGETS = LANG_ORDER.copy()

EXTRA_LANGS = {
    "mar": {"family": "Indo-European", "genus": "Indo-Aryan"},
    "ukr": {"family": "Indo-European", "genus": "Slavic"},
    "esp": {"family": "Indo-European", "genus": "Romance"},
    "ptbr": {"family": "Indo-European", "genus": "Romance"},
    "ptmz": {"family": "Indo-European", "genus": "Romance"},
    "ron": {"family": "Indo-European", "genus": "Romance"},
    "deu": {"family": "Indo-European", "genus": "Germanic"},
    "swe": {"family": "Indo-European", "genus": "Germanic"},
    "afr": {"family": "Indo-European", "genus": "Germanic"},
    "arq": {"family": "Afroasiatic", "genus": "Semitic"},
    "ary": {"family": "Afroasiatic", "genus": "Semitic"},
    "swa": {"family": "Niger-Congo", "genus": "Bantu"},
    "ibo": {"family": "Niger-Congo", "genus": "Volta-Niger"},
}

MODEL_NAME = "xlm-roberta-large"
LR = 5e-5
NUM_EPOCHS = 10
BATCH_SIZE = 8
MAX_LENGTH = 256
PATIENCE = 3
REDUCTION_FACTOR = 16

ADAPTER_CONFIG_LANG = SeqBnInvConfig(reduction_factor=REDUCTION_FACTOR)
ADAPTER_CONFIG_TASK = SeqBnConfig(reduction_factor=REDUCTION_FACTOR)
TASK_ADAPTER_NAME = "emotion_task"
RESULTS_PATH = f"{P3_DIR}/phase3_xlmr_validation_scores.json"

# False performs result validation and statistical analysis only.
RUN_MODEL_TRAINING = False

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Learning rate: {LR}")
print(f"Outputs: {P3_DIR}")

# ── Published benchmark reference and Phase 3 annotation constants ───────────
BENCHMARK_A = {
    "eng": 0.823, "hin": 0.926, "rus": 0.901, "hau": 0.751, "kin": 0.657,
    "sun": 0.550, "yor": 0.461, "vmw": 0.325, "pcm": 0.674,
}
BENCHMARK_C = {
    "eng": 0.797, "hin": 0.919, "rus": 0.906, "hau": 0.709, "kin": 0.519,
    "sun": 0.467, "yor": 0.359, "vmw": 0.210, "pcm": 0.674,
}
BENCHMARK_SRC_A = "SemEval-2025 Task 11, Table 5 (Track A)"
BENCHMARK_SRC_C = "SemEval-2025 Task 11, Table 7 (Track C)"
PHASE3_BENCHMARK_STATUS = "Context only: model score is validation; published reference is test"

Device: cuda
Model: xlm-roberta-large
Learning rate: 5e-05
Outputs: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3


In [ ]:
# Family/genus maps and C1/C2/C3 pool definitions
FAMILY = {c: v["family"] for c, v in {**LANGUAGES, **EXTRA_LANGS}.items()}
GENUS  = {c: v["genus"]  for c, v in {**LANGUAGES, **EXTRA_LANGS}.items()}

PCM_LEXIFIER_C2 = ["eng", "deu", "swe", "afr"]
PCM_LEXIFIER_C3 = ["eng"]


def get_c1_pool(target: str) -> list:
    return [c for c in ALL_TARGETS if c != target]


def get_c2_pool(target: str) -> list:
    if target == "pcm":
        return PCM_LEXIFIER_C2
    fam = FAMILY[target]
    return [c for c in FAMILY if FAMILY[c] == fam and c != target]


def get_c3_pool(target: str) -> list:
    if target == "pcm":
        return PCM_LEXIFIER_C3
    if target == "hau":
        return get_c2_pool("hau")
    if target == "sun":
        return []
    gen = GENUS[target]
    return [c for c in GENUS if GENUS[c] == gen and c != target]


print(f"{'Lang':6s} {'|C1|':>5s} {'|C2|':>5s} {'|C3|':>5s}  C3 pool")
print("-" * 60)
for code_ in RUN_TARGETS:
    c1, c2, c3 = get_c1_pool(code_), get_c2_pool(code_), get_c3_pool(code_)
    print(f"{code_:6s} {len(c1):5d} {len(c2):5d} {len(c3):5d}  {c3}")


Lang    |C1|  |C2|  |C3|  C3 pool
------------------------------------------------------------
eng        8    11     3  ['deu', 'swe', 'afr']
hin        8    11     1  ['mar']
rus        8    11     1  ['ukr']
hau        8     2     2  ['arq', 'ary']
kin        8     4     2  ['vmw', 'swa']
sun        8     0     0  []
yor        8     4     1  ['ibo']
vmw        8     4     2  ['kin', 'swa']
pcm        8     4     1  ['eng']


In [ ]:
# Data loading from the shared parquet cache
def get_emotion_cols(df: pd.DataFrame) -> list:
    return [e for e in EMOTION_ORDER if e in df.columns and df[e].fillna(0).sum() > 0]


def load_split(lang_code: str, split: str):
    cpath = f"{CACHE_DIR}/{lang_code}_{split}.parquet"
    if os.path.exists(cpath):
        return pd.read_parquet(cpath)
    try:
        ds = load_dataset(DATASET_HF_NAME, lang_code)
        split_key = split if split in ds else {"validation": "dev", "dev": "validation"}.get(split)
        if not split_key or split_key not in ds:
            return None
        df = ds[split_key].to_pandas()
        for e in EMOTION_ORDER:
            if e not in df.columns:
                df[e] = 0
        df.to_parquet(cpath)
        return df
    except Exception as exc:
        print(f"  Could not load {lang_code}/{split}: {exc}")
        return None


needed_extra = set()
for t in RUN_TARGETS:
    needed_extra |= set(get_c2_pool(t)) | set(get_c3_pool(t))
needed_extra -= set(ALL_TARGETS)
# C1 pools need every other BRIGHTER target too:
LANGS_TO_LOAD = ALL_TARGETS + sorted(needed_extra)

print(f"Loading {len(LANGS_TO_LOAD)} languages ...")
DATA = {}
for i, code_ in enumerate(LANGS_TO_LOAD, 1):
    is_run_target = code_ in RUN_TARGETS
    splits_needed = ["train", "validation", "test"] if is_run_target else ["train", "validation"]
    DATA[code_] = {}
    for split in splits_needed:
        df = load_split(code_, split)
        if df is not None:
            DATA[code_][split] = df
    print(f"  [{i:2d}/{len(LANGS_TO_LOAD)}] {code_} -- {list(DATA[code_].keys())}")
print("Data loaded.")


Loading 22 languages ...
  [ 1/22] eng -- ['train', 'validation', 'test']
  [ 2/22] hin -- ['train', 'validation', 'test']
  [ 3/22] rus -- ['train', 'validation', 'test']
  [ 4/22] hau -- ['train', 'validation', 'test']
  [ 5/22] kin -- ['train', 'validation', 'test']
  [ 6/22] sun -- ['train', 'validation', 'test']
  [ 7/22] yor -- ['train', 'validation', 'test']
  [ 8/22] vmw -- ['train', 'validation', 'test']
  [ 9/22] pcm -- ['train', 'validation', 'test']
  [10/22] afr -- ['train', 'validation']
  [11/22] arq -- ['train', 'validation']
  [12/22] ary -- ['train', 'validation']
  [13/22] deu -- ['train', 'validation']
  [14/22] esp -- ['train', 'validation']
  [15/22] ibo -- ['train', 'validation']
  [16/22] mar -- ['train', 'validation']
  [17/22] ptbr -- ['train', 'validation']
  [18/22] ptmz -- ['train', 'validation']
  [19/22] ron -- ['train', 'validation']
  [20/22] swa -- ['train', 'validation']
  [21/22] swe -- ['train', 'validation']
  [22/22] ukr -- ['train', 'validation']

In [ ]:
# MAD-X model, dataset, training, and evaluation
def _atomic_json_write(path: str, data) -> None:
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    os.sync()


def save_predictions(phase, lang, condition, y_true, y_pred, emotions):
    """Save per-sample predictions for significance tests and error analysis."""
    payload = {"emotions": list(emotions),
               "y_true": np.asarray(y_true).astype(int).tolist(),
               "y_pred": np.asarray(y_pred).astype(int).tolist()}
    path = f"{PRED_DIR}/{phase}_{lang}_{condition}.json"
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(payload, f); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    os.sync()


class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, emotion_cols, max_len=MAX_LENGTH):
        self.texts   = df["text"].tolist()
        self.labels  = df[emotion_cols].fillna(0).astype(float).values
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels":         torch.tensor(self.labels[idx], dtype=torch.float)}


def macro_f1(y_true, y_pred, emotions):
    scores = {e: round(f1_score(y_true[:, i], y_pred[:, i], zero_division=0), 4)
              for i, e in enumerate(emotions)}
    scores["macro_f1"] = round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4)
    return scores


def compute_class_weights(train_df, emotion_cols):
    """Calculate positive-class weights from the training data."""
    pos_counts = train_df[emotion_cols].fillna(0).sum()
    weights    = len(train_df) / (2 * pos_counts.clip(lower=1))
    return torch.tensor(weights.values, dtype=torch.float)


def build_madx_model(num_labels, lang_adapter_name, pretrained_lang_adapter_path=None):
    model = XLMRobertaAdapterModel.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)
    if pretrained_lang_adapter_path:
        model.load_adapter(pretrained_lang_adapter_path, config=ADAPTER_CONFIG_LANG,
                           load_as=lang_adapter_name)
    else:
        model.add_adapter(lang_adapter_name, config=ADAPTER_CONFIG_LANG)
    model.add_adapter(TASK_ADAPTER_NAME, config=ADAPTER_CONFIG_TASK)
    model.add_classification_head(TASK_ADAPTER_NAME, num_labels=num_labels, multilabel=True)
    stack = ac.Stack(lang_adapter_name, TASK_ADAPTER_NAME)
    model.set_active_adapters(stack)
    model.train_adapter(stack)
    return model.to(DEVICE)


def _adapter_head_state(model):
    return {k: v.cpu().clone() for k, v in model.state_dict().items()
            if ("adapters" in k or "invertible_adapters" in k
                or f"heads.{TASK_ADAPTER_NAME}" in k)}


def train_madx(model, train_df, dev_df, tokenizer, emotion_cols, ckpt_dir,
               epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
               use_pos_weight=False):
    gc.collect(); torch.cuda.empty_cache()

    train_dl = DataLoader(EmotionDataset(train_df, tokenizer, emotion_cols),
                          batch_size=batch_size, shuffle=True)
    dev_dl   = DataLoader(EmotionDataset(dev_df, tokenizer, emotion_cols),
                          batch_size=batch_size * 2)
    print(f"      train: {len(train_df)} ({len(train_dl)} batches/epoch)  dev: {len(dev_df)}")

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    opt   = AdamW(trainable_params, lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(0.1 * len(train_dl) * epochs),
        num_training_steps=len(train_dl) * epochs)

    # Positive-class weighting is used for the specified low-resource targets.
    # use_pos_weight=True for KIN/YOR/VMW (improved results); False for others
    if use_pos_weight:
        loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=compute_class_weights(train_df, emotion_cols).to(DEVICE))
    else:
        loss_fn = nn.BCEWithLogitsLoss()
    scaler = GradScaler()

    os.makedirs(ckpt_dir, exist_ok=True)
    state_path = f"{ckpt_dir}/state.json"
    last_path  = f"{ckpt_dir}/last_epoch.pt"
    best_path  = f"{ckpt_dir}/best.pt"
    opt_path   = f"{ckpt_dir}/opt.pt"
    sched_path = f"{ckpt_dir}/sched.pt"

    start_ep, best_f1, no_improve = 1, 0.0, 0
    if os.path.exists(state_path):
        try:
            with open(state_path) as f:
                st = json.load(f)
            start_ep, best_f1, no_improve = st["epoch"] + 1, st["best_f1"], st["no_improve"]
            _active_resume = model.active_adapters
            model.load_state_dict(torch.load(last_path, map_location=DEVICE), strict=False)
            if _active_resume is not None:
                model.set_active_adapters(_active_resume)
            opt.load_state_dict(torch.load(opt_path, map_location=DEVICE))
            sched.load_state_dict(torch.load(sched_path, map_location=DEVICE))
            print(f"      resuming from epoch {start_ep} (best dev F1: {best_f1:.4f})")
        except (json.JSONDecodeError, KeyError, ValueError):
            print("      corrupt checkpoint -- starting fresh")

    for ep in range(start_ep, epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_dl, desc=f"      epoch {ep}/{epochs}", leave=False)
        for step, batch in enumerate(pbar, 1):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["labels"].to(DEVICE)
            opt.zero_grad()
            with autocast():
                out  = model(input_ids=ids, attention_mask=mask, head=TASK_ADAPTER_NAME)
                loss = loss_fn(out.logits, lbl)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{running_loss/step:.4f}")

        dev_res = evaluate_madx(model, dev_df, tokenizer, emotion_cols,
                                batch_size=batch_size * 2)
        f1 = dev_res["macro_f1"]
        print(f"      epoch {ep}/{epochs} -- loss: {running_loss/len(train_dl):.4f} -- dev F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1, no_improve = f1, 0
            torch.save(_adapter_head_state(model), best_path)
        else:
            no_improve += 1

        torch.save(_adapter_head_state(model), last_path)
        torch.save(opt.state_dict(), opt_path)
        torch.save(sched.state_dict(), sched_path)
        with open(state_path, "w") as f:
            json.dump({"epoch": ep, "best_f1": best_f1, "no_improve": no_improve}, f)

        if no_improve >= patience:
            print(f"      early stop ({patience} epochs without improvement)")
            break

    _active = model.active_adapters

    ckpt = best_path if os.path.exists(best_path) else last_path
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE), strict=False)
    shutil.rmtree(ckpt_dir, ignore_errors=True)

    if _active is not None:
      model.set_active_adapters(_active)
    return model


@torch.no_grad()
def evaluate_madx(model, test_df, tokenizer, emotion_cols,
                  batch_size=BATCH_SIZE * 2, run_tag=None):
    test_dl = DataLoader(EmotionDataset(test_df, tokenizer, emotion_cols),
                         batch_size=batch_size)
    model.eval()
    preds_list, true_list = [], []
    for batch in test_dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask, head=TASK_ADAPTER_NAME)
        preds_list.append((torch.sigmoid(out.logits) > 0.5).int().cpu().numpy())
        true_list.append(batch["labels"].int().numpy())
    y_true_all, y_pred_all = np.vstack(true_list), np.vstack(preds_list)
    if run_tag is not None:
        save_predictions(run_tag[0], run_tag[1], run_tag[2],
                         y_true_all, y_pred_all, emotion_cols)
    return macro_f1(y_true_all, y_pred_all, emotion_cols)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)
print("XLM-R tokenizer and MAD-X functions ready.")

def _run_is_genuinely_complete(target, run_key, pred_tag):
    """
    Returns (is_complete, reason).
    Checks three things in order:
      1. Entry exists in the results JSON
      2. Prediction file exists on disk
      3. Adapter weights (lang + task) exist on disk
    N/A entries (empty pool / aliased) skip weight checks — no adapters were saved.
    Call this instead of the bare `run_key in phase3_results[target]` check so that
    a lost adapter directory triggers a re-run rather than a silent skip.
    """
    entry = phase3_results.get(target, {}).get(run_key)
    if entry is None:
        return False, "not in results JSON"

    # N/A or aliased — no adapter weights were ever saved for these
    if isinstance(entry, dict) and entry.get("macro_f1") is None:
        return True, "N/A (no training run)"
    if isinstance(entry, dict) and "aliased_from" in entry:
        return True, "aliased — no separate weights"

    # Prediction file
    pred_path = f"{PRED_DIR}/{pred_tag}_{target}_{run_key}.json"
    if not os.path.exists(pred_path):
        return False, f"prediction file missing: {os.path.basename(pred_path)}"

    # Adapter weights - both lang and task subdirs must have a pytorch_adapter.bin
    adir = f"{ADAPTER_DIR}/{target}_{run_key}"
    for sub in ["lang", "task"]:
        if not os.path.exists(f"{adir}/{sub}/pytorch_adapter.bin"):
            return False, f"adapter weights missing: {target}_{run_key}/{sub}/"

    return True, "complete"



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

XLM-R tokenizer and MAD-X functions ready.


In [ ]:
# Language-adaptive pre-training for Emakhuwa and Sundanese


from transformers import DataCollatorForLanguageModeling

class MLMTextDataset(Dataset):
    def __init__(self, sentences, tok, max_len=MAX_LENGTH):
        self.enc = tok(sentences, max_length=max_len, padding="max_length",
                       truncation=True, return_tensors="pt")
    def __len__(self):
        return self.enc["input_ids"].size(0)
    def __getitem__(self, idx):
        return {"input_ids":      self.enc["input_ids"][idx],
                "attention_mask": self.enc["attention_mask"][idx]}


def run_lapt(lang_code: str, sentences: list,
             epochs: int = 20, batch_size: int = 16, lr: float = 5e-5) -> str:
    adapter_name = f"lang_{lang_code}_lapt"
    adapter_path = f"{P3_DIR}/{adapter_name}"

    if os.path.exists(f"{adapter_path}/pytorch_adapter.bin"):
        print(f"  {lang_code} LAPT adapter found -- skipping retraining")
        return adapter_path

    print(f"  {len(sentences)} sentences for {lang_code} LAPT")
    mlm_dataset = MLMTextDataset(sentences, tokenizer)
    collator    = DataCollatorForLanguageModeling(
                      tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    mlm_dl      = DataLoader(mlm_dataset, batch_size=batch_size,
                             shuffle=True, collate_fn=collator)

    lapt_ckpt_dir   = f"{CKPT_DIR}/{lang_code}_lapt"
    os.makedirs(lapt_ckpt_dir, exist_ok=True)
    lapt_state_path = f"{lapt_ckpt_dir}/state.json"
    lapt_last_path  = f"{lapt_ckpt_dir}/last_epoch.pt"


    lapt_model = XLMRobertaAdapterModel.from_pretrained(
                     MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)
    lapt_model.add_adapter(adapter_name, config=ADAPTER_CONFIG_LANG)
    lapt_model.add_masked_lm_head(adapter_name)
    lapt_model.set_active_adapters(adapter_name)
    lapt_model.train_adapter(adapter_name)
    lapt_model.to(DEVICE)

    opt    = AdamW([p for p in lapt_model.parameters() if p.requires_grad], lr=lr)
    scaler = GradScaler()

    start_ep = 1
    if os.path.exists(lapt_state_path):
        try:
            with open(lapt_state_path) as f:
                st = json.load(f)
            start_ep = st["epoch"] + 1
            lapt_model.load_state_dict(
                torch.load(lapt_last_path, map_location=DEVICE), strict=False)
            print(f"  resuming {lang_code} LAPT from epoch {start_ep}")
        except (json.JSONDecodeError, KeyError, RuntimeError):
            print(f"  corrupt LAPT checkpoint -- restarting {lang_code}")
            start_ep = 1

    print(f"  Training -- {len(mlm_dl)} batches/epoch x {epochs} epochs")
    for ep in range(start_ep, epochs + 1):
        lapt_model.train()
        running = 0.0
        pbar = tqdm(mlm_dl, desc=f"  {lang_code} LAPT {ep}/{epochs}", leave=False)
        for step, batch in enumerate(pbar, 1):
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            opt.zero_grad()
            with autocast():
                out  = lapt_model(input_ids=ids, attention_mask=mask, labels=labels)
                loss = out.loss
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            running += loss.item()
            pbar.set_postfix(loss=f"{running/step:.4f}")

        print(f"  {lang_code} LAPT epoch {ep}/{epochs} -- loss: {running/len(mlm_dl):.4f}")
        torch.save(lapt_model.state_dict(), lapt_last_path)
        with open(lapt_state_path, "w") as f:
            json.dump({"epoch": ep}, f)

    os.makedirs(adapter_path, exist_ok=True)
    lapt_model.save_adapter(adapter_path, adapter_name)
    shutil.rmtree(lapt_ckpt_dir, ignore_errors=True)
    del lapt_model; gc.collect(); torch.cuda.empty_cache()
    print(f"  {lang_code} LAPT complete -- saved to {adapter_path}")
    return adapter_path

# Reuse the final adapters unless training is explicitly enabled.
vmw_lapt_path = f"{P3_DIR}/lang_vmw_lapt"
sun_lapt_path = f"{P3_DIR}/lang_sun_lapt"

if RUN_MODEL_TRAINING:
    vmw_sentences = DATA["vmw"]["train"]["text"].dropna().tolist()
    vmw_lapt_path = run_lapt("vmw", vmw_sentences)

    sun_sentences = []
    for source_code in ["jav", "ind"]:
        for split in ["validation", "test"]:
            source_df = load_split(source_code, split)
            if source_df is not None and "text" in source_df.columns:
                sun_sentences.extend(source_df["text"].dropna().tolist())
    sun_lapt_path = run_lapt("sun", sun_sentences)

LAPT_PATHS = {
    "vmw": vmw_lapt_path,
    "sun": sun_lapt_path,
}

if not RUN_MODEL_TRAINING:
    missing_adapters = [
        path for path in LAPT_PATHS.values()
        if not os.path.exists(f"{path}/pytorch_adapter.bin")
    ]
    if missing_adapters:
        print("Saved LAPT adapters not found; they are only required for retraining:")
        for path in missing_adapters:
            print(f"  {path}")

print(f"LAPT adapter paths: {LAPT_PATHS}")


LAPT adapter paths: {'vmw': '/content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/lang_vmw_lapt', 'sun': '/content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/lang_sun_lapt'}


In [ ]:
if RUN_MODEL_TRAINING:
    # Main Phase 3 XLM-R experiment loop

    POS_WEIGHT_LANGS = {"kin", "yor", "vmw"}

    if os.path.exists(RESULTS_PATH):
        try:
            with open(RESULTS_PATH) as f:
                phase3w_results = json.load(f)
            print(f"Resuming -- {sum(len(v) for v in phase3w_results.values())} runs saved")
        except (json.JSONDecodeError, ValueError):
            print("Corrupt results file -- starting fresh")
            phase3w_results = {}
    else:
        phase3w_results = {}
        print("Starting fresh (matched recipe).")

    CONFIG_FNS = {"C1": get_c1_pool, "C2": get_c2_pool, "C3": get_c3_pool}

    for target in RUN_TARGETS:
        phase3w_results.setdefault(target, {})

        ecols   = get_emotion_cols(DATA[target]["train"])
        test_df = DATA[target]["validation"]

        for track in ["track_a", "track_c"]:
            for cfg_name, pool_fn in CONFIG_FNS.items():
                run_key = f"{track}_{cfg_name}"

                # ── Skip / inference-only / re-run guard ─────────────────
                # 1. pred + weights both present  -> skip (backfill JSON if lost)
                # 2. weights present, pred missing -> inference-only, no retraining
                # 3. pred present, weights missing -> retrain to recover weights
                # 4. neither                       -> train from scratch
                _pred_path    = f"{PRED_DIR}/phase3xlmrW_validation_{target}_{run_key}.json"
                _adapter_path = f"{ADAPTER_DIR}/{target}_{run_key}"
                _pred_ok      = os.path.exists(_pred_path)
                _adapter_ok   = os.path.exists(f"{_adapter_path}/lang/pytorch_adapter.bin")
                if _pred_ok and _adapter_ok:
                    if run_key not in phase3w_results[target]:
                        _yt, _yp, _ = load_align_prediction(_pred_path, rewrite=False)
                        phase3w_results[target][run_key] = score_saved_prediction(_yt, _yp, target)
                        _atomic_json_write(RESULTS_PATH, phase3w_results)
                    print(f"  Skipping {LANGUAGES[target]['name']} / {run_key} -- predictions + adapters intact")
                    continue
                elif _adapter_ok and not _pred_ok:
                    print(f"  Inference-only {LANGUAGES[target]['name']} / {run_key} -- adapter found, generating prediction")
                    _lang_inf = f"lang_{target}_{run_key}_inf"
                    _m = XLMRobertaAdapterModel.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)
                    _m.load_adapter(f"{_adapter_path}/lang", load_as=_lang_inf)
                    _m.load_adapter(f"{_adapter_path}/task", load_as=TASK_ADAPTER_NAME)
                    _m.load_head(f"{_adapter_path}/head")
                    _m.set_active_adapters(ac.Stack(_lang_inf, TASK_ADAPTER_NAME))
                    _m = _m.to(DEVICE); _m.eval()
                    _res = evaluate_madx(_m, test_df, tokenizer, ecols,
                                         run_tag=("phase3xlmrW_validation", target, run_key))
                    phase3w_results[target][run_key] = _res
                    _atomic_json_write(RESULTS_PATH, phase3w_results)
                    print(f"    macro F1 = {_res['macro_f1']:.4f}")
                    del _m; gc.collect(); torch.cuda.empty_cache()
                    continue
                elif _pred_ok and not _adapter_ok:
                    print(f"  Re-running {LANGUAGES[target]['name']} / {run_key} -- adapter weights missing from Drive")
                    phase3w_results[target].pop(run_key, None)

                source_pool = pool_fn(target)

                # Empty family/genus pool -> N/A, regardless of track. Without
                # this guard, Track A would silently fall back to training on
                # the target language alone (train_langs = [target]) and save
                # that as if it were a genuine C2/C3 cross-lingual result.
                if not source_pool and cfg_name != "C1":
                    phase3w_results[target][run_key] = {
                        "macro_f1": None,
                        "note": "N/A -- empty family/genus pool (no Arabic data used; "
                                "Hausa is the only Afroasiatic BRIGHTER target)"}
                    _atomic_json_write(RESULTS_PATH, phase3w_results)
                    print(f"  N/A (empty pool): {LANGUAGES[target]['name']} / {run_key}")
                    continue

                # C3 == C2 -> alias, never retrain
                if cfg_name == "C3":
                    c2_key = f"{track}_C2"
                    if (source_pool and set(source_pool) == set(get_c2_pool(target))
                            and c2_key in phase3w_results[target]):
                        phase3w_results[target][run_key] = {
                            **phase3w_results[target][c2_key], "aliased_from": c2_key}
                        _atomic_json_write(RESULTS_PATH, phase3w_results)
                        print(f"  Aliased {LANGUAGES[target]['name']} / {run_key} -> {c2_key}")
                        continue

                train_langs = list(source_pool) + ([target] if track == "track_a" else [])

                valid_train = [c for c in train_langs if c in DATA and "train" in DATA[c]]
                if not valid_train:
                    phase3w_results[target][run_key] = {"macro_f1": None,
                                                        "note": "N/A -- empty source pool"}
                    _atomic_json_write(RESULTS_PATH, phase3w_results)
                    print(f"  N/A: {LANGUAGES[target]['name']} / {run_key}")
                    continue

                print(f"\n  [{target.upper()}] {LANGUAGES[target]['name']} -- {run_key}"
                      f" -- sources: {valid_train}")

                train_df = pd.concat([DATA[c]["train"] for c in valid_train], ignore_index=True)
                dev_df   = pd.concat([DATA[c]["validation"] for c in valid_train
                                      if "validation" in DATA[c]], ignore_index=True)

                lang_adapter_name = f"lang_{target}_{cfg_name}_w"

                # Deterministic per-run seed.
                _run_seed = RANDOM_SEED + hash(f"{target}_{run_key}") % 10000
                random.seed(_run_seed); np.random.seed(_run_seed)
                torch.manual_seed(_run_seed); torch.cuda.manual_seed_all(_run_seed)

                model = build_madx_model(len(ecols), lang_adapter_name,
                                         pretrained_lang_adapter_path=LAPT_PATHS.get(target))
                _use_pw = target in POS_WEIGHT_LANGS
                print(f"      Loss: {'pos_weight BCE' if _use_pw else 'standard BCE'}")
                model = train_madx(model, train_df, dev_df, tokenizer, ecols,
                                   ckpt_dir=f"{CKPT_DIR}/{target}_{run_key}",
                                   use_pos_weight=_use_pw)

                res   = evaluate_madx(model, test_df, tokenizer, ecols,
                                      run_tag=("phase3xlmrW_validation", target, run_key))

                phase3w_results[target][run_key] = res
                _atomic_json_write(RESULTS_PATH, phase3w_results)
                # print(f"    Results saved -- {run_key} macro F1 = {res['macro_f1']:.4f}")

                # Then save adapters (can stall on Drive, but results are already safe)
                adir = f"{ADAPTER_DIR}/{target}_{run_key}"
                os.makedirs(adir, exist_ok=True)
                model.save_adapter(f"{adir}/lang", lang_adapter_name)
                model.save_adapter(f"{adir}/task", TASK_ADAPTER_NAME)
                model.save_head(f"{adir}/head", TASK_ADAPTER_NAME)
                os.sync()
                print(f"    Saved -- {run_key} macro F1 = {res['macro_f1']:.4f}"
                      f"  (adapters -> {adir})")

                del model
                gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

    print("\n" + "=" * 60)
    print("Phase 3 XLM-R run complete.")
    print(f"Results: {RESULTS_PATH}")
else:
    print('Training disabled; saved XLM-R predictions will be validated and analysed.')


Training disabled; saved XLM-R predictions will be validated and analysed.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Validate saved predictions, rebuild aggregate scores, and run paired bootstrap tests.
# ─────────────────────────────────────────────────────────────────────────────

def _atomic_json_write_safe(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(tmp, path)


def load_align_prediction(path, rewrite=True):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    required = {"emotions", "y_true", "y_pred"}
    missing_keys = required - set(payload)
    if missing_keys:
        raise ValueError(f"{path}: missing keys {sorted(missing_keys)}")
    emotions = list(payload["emotions"])
    if len(emotions) != len(set(emotions)):
        raise ValueError(f"{path}: duplicated emotion names {emotions}")

    y_true_raw = np.asarray(payload["y_true"], dtype=int)
    y_pred_raw = np.asarray(payload["y_pred"], dtype=int)
    if y_true_raw.ndim != 2 or y_pred_raw.ndim != 2 or y_true_raw.shape != y_pred_raw.shape:
        raise ValueError(f"{path}: incompatible arrays y_true={y_true_raw.shape}, y_pred={y_pred_raw.shape}")
    if y_true_raw.shape[1] != len(emotions):
        raise ValueError(f"{path}: columns do not match stored emotion names")

    if set(emotions) == set(EMOTION_ORDER):
        # Normal case - all 6 emotions present, just reorder if needed
        indices = [emotions.index(e) for e in EMOTION_ORDER]
        y_true = y_true_raw[:, indices]
        y_pred = y_pred_raw[:, indices]
        changed = (emotions != EMOTION_ORDER)

    elif set(emotions) == set(EMOTION_ORDER) - {"disgust"}:
        # English case - disgust is structurally absent, inject zero column
        n = y_true_raw.shape[0]
        y_true = np.zeros((n, len(EMOTION_ORDER)), dtype=int)
        y_pred = np.zeros((n, len(EMOTION_ORDER)), dtype=int)
        for old_idx, emo in enumerate(emotions):
            new_idx = EMOTION_ORDER.index(emo)
            y_true[:, new_idx] = y_true_raw[:, old_idx]
            y_pred[:, new_idx] = y_pred_raw[:, old_idx]
        changed = True  # always rewrite to upgrade to canonical 6-column format

    else:
        raise ValueError(f"{path}: expected {EMOTION_ORDER}, found {emotions}")

    if changed and rewrite:
        _atomic_json_write_safe(path, {
            "emotions": EMOTION_ORDER,
            "y_true": y_true.tolist(),
            "y_pred": y_pred.tolist(),
        })
        print(f"Aligned prediction to 6-emotion format: {os.path.basename(path)}")

    return y_true, y_pred, list(EMOTION_ORDER)


def score_saved_prediction(y_true, y_pred, lang_code):
    active = [e for e in EMOTION_ORDER if not (lang_code == "eng" and e == "disgust")]
    output = {}
    unrounded = {}
    for index, emotion in enumerate(EMOTION_ORDER):
        value = f1_score(y_true[:, index], y_pred[:, index], zero_division=0)
        unrounded[emotion] = float(value)
        output[emotion] = round(float(value), 4)
    output["macro_f1"] = round(float(np.mean([unrounded[e] for e in active])), 4)
    return output

from itertools import combinations
import zlib

N_BOOT = 10_000
BOOT_CHUNK = 64
BASE_SEED = 42
ALPHA = 0.05


def _stable_seed(label):
    return int((BASE_SEED + zlib.crc32(label.encode("utf-8"))) % (2**32 - 1))


def _active_indices(lang):
    return [i for i,e in enumerate(EMOTION_ORDER) if not (lang == "eng" and e == "disgust")]


def _bootstrap_scores(y_true, systems, lang, seed_label):
    active = _active_indices(lang)
    y = np.asarray(y_true[:, active], dtype=np.int8)
    names = list(systems)
    blocks = []
    for name in names:
        pred = np.asarray(systems[name][:, active], dtype=np.int8)
        if pred.shape != y.shape:
            raise ValueError(f"{seed_label}/{name}: {pred.shape} != {y.shape}")
        tp = ((y == 1) & (pred == 1)).astype(np.uint8)
        fp = ((y == 0) & (pred == 1)).astype(np.uint8)
        fn = ((y == 1) & (pred == 0)).astype(np.uint8)
        blocks.append(np.stack([tp, fp, fn], axis=2))
    contributions = np.stack(blocks, axis=1)
    flat = contributions.reshape(len(y), -1)
    n_systems, n_labels = len(names), len(active)

    def _macro(counts):
        tp, fp, fn = counts[...,0], counts[...,1], counts[...,2]
        den = 2*tp + fp + fn
        f1 = np.divide(2*tp, den, out=np.zeros_like(den,dtype=float), where=den!=0)
        return np.mean(f1, axis=-1)

    obs_counts = flat.sum(axis=0,dtype=np.int64).reshape(n_systems,n_labels,3)
    obs_values = _macro(obs_counts)
    observed = {n: float(obs_values[i]) for i,n in enumerate(names)}
    boot_matrix = np.empty((N_BOOT,n_systems),dtype=float)
    rng = np.random.default_rng(_stable_seed(seed_label))
    pos=0
    while pos < N_BOOT:
        size=min(BOOT_CHUNK,N_BOOT-pos)
        idx=rng.integers(0,len(y),size=(size,len(y)),dtype=np.int32)
        counts=flat[idx].sum(axis=1,dtype=np.int64).reshape(size,n_systems,n_labels,3)
        boot_matrix[pos:pos+size]=_macro(counts)
        pos += size
    return observed,{n:boot_matrix[:,i] for i,n in enumerate(names)}


def _summarise(values, observed):
    values=np.asarray(values,dtype=float)
    p=min(2*min((np.sum(values<=0)+1)/(len(values)+1),
                (np.sum(values>=0)+1)/(len(values)+1)),1.0)
    lo,hi=np.percentile(values,[2.5,97.5])
    return {"delta":float(observed),"ci_low":float(lo),"ci_high":float(hi),"p":float(p)}


def _holm(pvalues):
    p=np.asarray(pvalues,dtype=float); m=len(p)
    order=np.argsort(p); adjusted=np.empty(m); running=0.0
    for rank,idx in enumerate(order):
        running=max(running,min(1.0,(m-rank)*p[idx])); adjusted[idx]=running
    return adjusted


def _finish_family(rows, expected, output_path, family):
    frame = pd.DataFrame(rows)
    if len(frame) != expected:
        raise ValueError(f"{family}: expected {expected} tests, found {len(frame)}")
    frame["Holm-adjusted p-value"] = _holm(frame["Raw p-value"].astype(float).values)
    frame["Significant after Holm correction (alpha=0.05)"] = frame["Holm-adjusted p-value"] < ALPHA
    frame["Evaluation split"] = "Validation"
    frame["Statistical test"] = "Paired bootstrap; Holm correction applied within the stated comparison family"
    frame.to_csv(output_path, index=False)
    print(f"Saved {len(frame)} tests: {output_path}")
    return frame

RESULTS_PATH = f"{P3_DIR}/phase3_xlmr_validation_scores.json"
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as _f:
        corrected_results=json.load(_f)
else:
    corrected_results={}

for lang in LANG_ORDER:
    corrected_results.setdefault(lang,{})
    for track in ["a","c"]:
        for cfg in ["C1","C2","C3"]:
            path=f"{PRED_DIR}/phase3xlmrW_validation_{lang}_track_{track}_{cfg}.json"
            if not os.path.exists(path):
                continue
            y_true,y_pred,_=load_align_prediction(path,rewrite=True)
            key=f"track_{track}_{cfg}"
            prior=corrected_results[lang].get(key,{})
            score=score_saved_prediction(y_true,y_pred,lang)
            if isinstance(prior,dict):
                score={**prior,**score}
            corrected_results[lang][key]=score
_atomic_json_write_safe(RESULTS_PATH,corrected_results)
print(f"Validated Phase 3 XLM-R results: {RESULTS_PATH}")

rows=[]
for lang in LANG_ORDER:
    for track in ["a","c"]:
        systems={}; reference=None
        for cfg in ["C1","C2","C3"]:
            path=f"{PRED_DIR}/phase3xlmrW_validation_{lang}_track_{track}_{cfg}.json"
            if not os.path.exists(path): continue
            y,p,_=load_align_prediction(path,rewrite=False)
            if reference is None: reference=y
            elif not np.array_equal(reference,y): raise ValueError(f"{lang}/{track}/{cfg}: gold mismatch")
            systems[cfg]=p
        if len(systems)<2: continue
        observed,boot=_bootstrap_scores(reference,systems,lang,f"phase3_xlmr_{lang}_{track}")
        for a,b in combinations(["C1","C2","C3"],2):
            if a not in systems or b not in systems: continue
            stat=_summarise(boot[a]-boot[b],observed[a]-observed[b])
            rows.append({
                "Language code":                              lang.upper(),
                "Language":                                   LANGUAGES[lang]["name"],
                "Evaluation track":                           f"Track {track.upper()}",
                "Comparison":                                 f"{a} vs {b}",
                "First transfer configuration":               a,
                "Second transfer configuration":              b,
                "First-condition macro-F1":                   observed[a],
                "Second-condition macro-F1":                  observed[b],
                "Macro-F1 difference (first minus second)":   stat["delta"],
                "95% confidence interval lower bound":        stat["ci_low"],
                "95% confidence interval upper bound":        stat["ci_high"],
                "Raw p-value":                                stat["p"],
                "Higher-scoring condition":                   a if stat["delta"] > 0 else b,
            })
# phase3_xlmr_significance=_finish_family(rows,44,f"{P3_DIR}/phase3_xlmr_significance.csv","Phase 3 XLM-R")
phase3_xlmr_significance = _finish_family(rows, len(rows),f"{P3_DIR}/phase3_xlmr_significance.csv","Phase 3 XLM-R")

Validated Phase 3 XLM-R results: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/phase3_xlmr_validation_scores.json
Saved 42 tests: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/phase3_xlmr_significance.csv


In [ ]:
def _safe_load(path):
    if not os.path.exists(path):
        print(f"Warning: Required file not found: {path}. Defaulting to N/A.")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

xlmr_results      = _safe_load(f"{P3_DIR}/phase3_xlmr_validation_scores.json")
serengeti_results = _safe_load(f"{P3_DIR}/phase3_serengeti_validation.json")

SERENGETI_LANGS = ["hau", "kin", "yor", "vmw", "pcm"]
LANGUAGE_NAMES  = {
    "hau": "Hausa", "kin": "Kinyarwanda", "yor": "Yoruba",
    "vmw": "Emakhuwa", "pcm": "Nigerian Pidgin",
}

rows = []
for code in SERENGETI_LANGS:
    for cfg in ["C1", "C2", "C3"]:
        for track, track_label in [("track_a", "A"), ("track_c", "C")]:
            result_key = f"{track}_{cfg}"

            # Safely fetch dictionaries, defaulting to empty if not found or malformed
            xlmr_lang = xlmr_results.get(code) if isinstance(xlmr_results.get(code), dict) else {}
            serengeti_lang = serengeti_results.get(code) if isinstance(serengeti_results.get(code), dict) else {}

            xlmr_result = xlmr_lang.get(result_key) if isinstance(xlmr_lang.get(result_key), dict) else {}
            serengeti_result = serengeti_lang.get(result_key) if isinstance(serengeti_lang.get(result_key), dict) else {}

            xlmr_score      = xlmr_result.get("macro_f1")
            serengeti_score = serengeti_result.get("macro_f1")

            # Skip only if BOTH are missing so we can still track incomplete runs
            if xlmr_score is None and serengeti_score is None:
                continue

            # Process XLM-R score
            if xlmr_score is not None:
                xlmr_score = float(xlmr_score)
                xlmr_disp = round(xlmr_score, 4)
            else:
                xlmr_disp = "N/A"

            # Process SERENGETI score
            if serengeti_score is not None:
                serengeti_score = float(serengeti_score)
                serengeti_disp = round(serengeti_score, 4)
            else:
                serengeti_disp = "N/A"

            # Calculate winner and backbone effect only if both scores exist
            if xlmr_score is not None and serengeti_score is not None:
                effect = round(serengeti_score - xlmr_score, 4)
                if serengeti_score > xlmr_score:
                    winner = "SERENGETI"
                elif xlmr_score > serengeti_score:
                    winner = "XLM-R"
                else:
                    winner = "Tie"
            else:
                effect = "N/A"
                winner = "N/A"

            rows.append({
                "Language code":                                        code.upper(),
                "Language":                                             LANGUAGE_NAMES[code],
                "Evaluation track":                                     f"Track {track_label}",
                "Transfer configuration":                               cfg,
                "XLM-R macro-F1":                                       xlmr_disp,
                "SERENGETI macro-F1":                                   serengeti_disp,
                "Backbone effect (SERENGETI minus XLM-R)":              effect,
                "Higher-scoring condition":                             winner,
                "Evaluation split":                                     "Validation",
                "Published Track A test best macro-F1 (context only)":  BENCHMARK_A.get(code.lower()),
                "Published Track C test best macro-F1 (context only)":  BENCHMARK_C.get(code.lower()),
                "Published test best for this row's track (context only)": BENCHMARK_A.get(code.lower()) if track_label == "A" else BENCHMARK_C.get(code.lower()),
                "Published benchmark source":                           BENCHMARK_SRC_A if track_label == "A" else BENCHMARK_SRC_C,
                "Benchmark comparison status":                          PHASE3_BENCHMARK_STATUS,
            })

df_cmp = pd.DataFrame(rows)

# Expected 26 when both tracks complete for all 5 languages.
if len(df_cmp) != 26:
    print(f"Warning: expected 26 comparisons, found {len(df_cmp)} (subset or incomplete run).")

# Assertion passes safely because pandas treats the string "N/A" as valid object data rather than NaN/None
assert not df_cmp[["XLM-R macro-F1", "SERENGETI macro-F1", "Backbone effect (SERENGETI minus XLM-R)"]].isna().any().any()

print(df_cmp.to_string(index=False))

out         = f"{P3_DIR}/phase3_backbone_comparison.csv"
tmp_out     = f"{out}.tmp"
df_cmp.to_csv(tmp_out, index=False)
os.replace(tmp_out, out)

print(f"\nSaved to: {out}")
print(f"Controlled comparisons: {len(df_cmp)}")
print("\nWinner counts:")
print(df_cmp["Higher-scoring condition"].value_counts().to_string())
print("\n✓ XLM-R and SERENGETI compared (missing data handled safely)")
print("✓ Backbone effects calculated")
print("✓ Comparison export complete")

Language code        Language Evaluation track Transfer configuration XLM-R macro-F1 SERENGETI macro-F1 Backbone effect (SERENGETI minus XLM-R) Higher-scoring condition Evaluation split  Published Track A test best macro-F1 (context only)  Published Track C test best macro-F1 (context only)  Published test best for this row's track (context only)              Published benchmark source                                          Benchmark comparison status
          HAU           Hausa          Track A                     C1         0.6792             0.6974                                  0.0182                SERENGETI       Validation                                                0.751                                                0.709                                                    0.751 SemEval-2025 Task 11, Table 5 (Track A) Context only: model score is validation; published reference is test
          HAU           Hausa          Track C                     C1         0.3016

In [ ]:
import pandas as pd
import json

DRIVE = "/content/drive/MyDrive/EmotionDetection/CAST_checkpoints"
P3_PATH = f"{DRIVE}/Phase3/phase3_xlmr_validation_scores.json"
P4_PATH = f"{DRIVE}/Phase4/phase4_validation_best_system.csv"
P4_SRNG = f"{DRIVE}/Phase4/phase4_serengeti_test_ablation_results.json"

with open(P3_PATH) as f:
    p3 = json.load(f)
with open(P4_SRNG) as f:
    srng = json.load(f)

df = pd.read_csv(P4_PATH)

# Build XLM-R val lookup from Phase 3 validation scores
def get_xlmr_val(row):
    lang = row["Language code"].lower()      # Phase 4 now stores uppercase codes
    cfg  = row["Validation-selected XLM-R configuration"]  # renamed in Phase 4
    try:
        return p3[lang][cfg.upper()]["macro_f1"]
    except (KeyError, TypeError):
        return None

df["Best XLM-R validation macro-F1"] = df.apply(get_xlmr_val, axis=1)

# Fill SERENGETI CAST test score from Phase 4 SERENGETI results (Tier 2/3 only)
srng_cast = {}
for lang, data in srng.items():
    srng_cast[lang] = data.get("both", {}).get("macro_f1", None)

def get_srng_cast(row):
    return srng_cast.get(row["Language code"].lower(), None)

df["Gemma CAST test macro-F1 using SERENGETI-selected configuration"] = df.apply(get_srng_cast, axis=1)

df.to_csv(P4_PATH, index=False)
print(df[[
    "Language",
    "Best XLM-R validation macro-F1",
    "Gemma CAST test macro-F1 using XLM-R-selected configuration",
    "Gemma CAST test macro-F1 using SERENGETI-selected configuration",
    "Highest-scoring Phase 4 condition",
    "Highest Phase 4 test macro-F1",
]].to_string(index=False))

       Language  Best XLM-R validation macro-F1  Gemma CAST test macro-F1 using XLM-R-selected configuration  Gemma CAST test macro-F1 using SERENGETI-selected configuration Highest-scoring Phase 4 condition  Highest Phase 4 test macro-F1
        English                          0.7634                                                       0.6290                                                              NaN                    G4-CAST(XLM-R)                         0.6290
          Hindi                          0.8433                                                       0.8496                                                              NaN                    G4-CAST(XLM-R)                         0.8496
        Russian                             NaN                                                       0.8559                                                              NaN                    G4-CAST(XLM-R)                         0.8559
          Hausa                             